### Transform Races Data

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.races"
silver_table = f"{catalog_name}.{silver_schema}.races"

#### Step 1 - Reading the data into dataframe

In [0]:
races_df = spark.table(bronze_table)

####Step 2 - Drop URL column as it is not needed for analytics

In [0]:
from pyspark.sql import functions as F

In [0]:
races_selected_df = races_df.select(
    F.col("season"),
    F.col("round"),
    F.col("raceName"),
    F.col("date"),
    F.col("circuitId"),
    F.col("ingestion_timestamp"),
    F.col("source")
)

####Step 3 & 4 - Standardize Column Names
 - Standardize column names using snake_case(raceName to race_name, circuitId to circuit_id)
 - Rename columns to make them more meaningful(date to race_date)

In [0]:
races_renamed_df = races_selected_df.withColumnsRenamed({
    "raceName" : "race_name",
    "circuitId" : "circuit_id",
    "date" : "race_date"
})

####Step 5 - Just checking if there are any null primary values

In [0]:
%sql
select count(*) from formula1.bronze.races where season is NULL or round is NULL

#### Step 6 - Remove duplicate records

In [0]:
races_distinct_df = races_renamed_df.dropDuplicates(["season", "round"])

####Step 7 - Transform values of race_name to Title Case

In [0]:
races_final_df = races_distinct_df.withColumn('race_name', F.initcap(F.col("race_name")))

####Step 8 - Write the transformed data into Silver races table

In [0]:
(
    races_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
)